# TP4 : Distillation de Modèles de Raisonnement (DASD)
## Distribution-Aligned Sequence Distillation — Application aux Échecs

**Domaine choisi** : Théorie des échecs (ouvertures, stratégies, finales)  
**Modèle enseignant (Teacher)** : `openai/gpt-oss-120b` via API Infomaniak  
**Modèle étudiant (Student)** : `Qwen3-4B` / `unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit`

---
## Phase 1 : Installation de l'environnement

In [ ]:
!nvidia-smi

Sun Mar  1 11:36:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install "llamafactory[torch,bitsandbytes]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.5/398.5 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!llamafactory-cli version

2026-03-01 11:37:33.659200: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772365053.720739     567 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772365053.754014     567 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772365053.829826     567 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772365053.829883     567 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772365053.829896     567 computation_placer.cc:177] computation placer alr

In [ ]:
!pip install datasets openai nltk

---
## Phase 2 : Étude du dataset de référence

Exploration du dataset officiel DASD sur HuggingFace pour comprendre le format attendu (instruction/response avec tags `<reasoning>`).

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_tree(repo_id="Alibaba-Apsara/Superior-Reasoning-SFT-gpt-oss-120b", repo_type="dataset")
for f in files:
  print(f)


RepoFolder(path='assets', tree_id='eb8390b8a7e50a222934df2bb5131e2c1890df77', last_commit=None)
RepoFile(path='.gitattributes', size=2787, blob_id='c1df2bd0e889c27bf11c60556cfc7c40804ec3b0', lfs=None, last_commit=None, security=None)
RepoFile(path='README.md', size=11811, blob_id='22b5d593248ae8fe316424b17d1a642bd933b943', lfs=None, last_commit=None, security=None)
RepoFile(path='Superior-Reasoning-SFT-gpt-oss-120b-stage1-train-data.jsonl', size=4601583540, blob_id='5c4f5ff8d6cc80f029fb7cbf55a182b464142af3', lfs=BlobLfsInfo(size=4601583540, sha256='8e8ea9eebebcaa9220a8c6b47c0417757dbb001821fb64f06269c5256d0ebbe7', pointer_size=135), last_commit=None, security=None)
RepoFile(path='Superior-Reasoning-SFT-gpt-oss-120b-stage2-train-data.jsonl', size=20167603292, blob_id='42b62a5682df040a4e9b39ba17a06842af5f518a', lfs=BlobLfsInfo(size=20167603292, sha256='adfc8b444ea780e8d6c2c2c97b4a470ac8cfdab298bc08f45d631966c38b8b29', pointer_size=136), last_commit=None, security=None)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import requests
import json

url = "https://huggingface.co/datasets/Alibaba-Apsara/Superior-Reasoning-SFT-gpt-oss-120b/resolve/main/Superior-Reasoning-SFT-gpt-oss-120b-stage1-train-data.jsonl"

response = requests.get(url, headers={"Range": "bytes=0-50000"}, stream=True)
lines = response.text.strip().split("\n")

print(f"Lignes récupérées : {len(lines)}")
for i, line in enumerate(lines[:2]):
    example = json.loads(line)
    print(f"\n{'='*80}")
    print(f"Exemple {i} - Clés : {list(example.keys())}")
    for key, value in example.items():
        text = json.dumps(value, ensure_ascii=False) if not isinstance(value, str) else value
        print(f"\n[{key}]: {text[:500]}")

Lignes récupérées : 2

Exemple 0 - Clés : ['uuid', 'input', 'output', 'domain', 'meta']

[uuid]: a7c94c27-1c6a-434a-bec9-dd90c1acac28

[input]: Find the sum of the first six terms of an arithmetic sequence in which the sum of any number of terms is equal to four times the square of that number. Let's think step by step and output the final answer within \boxed{}.

[output]: <think>
We need to parse the problem: "Find the sum of the first six terms of an arithmetic sequence in which the sum of any number of terms is equal to four times the square of that number." So we have an arithmetic sequence (i.e., terms are a, a+d, a+2d, ...). The sum of the first n terms S_n is equal to 4 * n^2 (i.e., S_n = 4n^2) for any n (presumably positive integer). We need to find S_6, the sum of the first six terms. But wait, if S_n = 4n^2, then S_6 = 4*36 = 144. However, that seems too

[domain]: math

[meta]: {"training_stage": "stage1", "sampling_temperature": 0.6, "teacher_model": "gpt-oss-120b", "logpr

JSONDecodeError: Unterminated string starting at: line 1 column 442 (char 441)

---
## Phase 3 : Génération du dataset via API

### 3.1 Configuration et test de l'API Infomaniak

In [ ]:
# TEST de la clé.

import openai

client = openai.OpenAI(
  base_url="https://api.infomaniak.com/2/ai/48/openai/v1",
  api_key="nKuJabWS1epvq3x-m8by6NOU4xP4_znNL9OhmgXBPz9OeWOHlyGJIENnG8oXLT-4oOXNmESqExEMZv6o"
)

TEACHER_MODEL = "openai/gpt-oss-120b"


response = client.chat.completions.create(
    model=TEACHER_MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a chess grandmaster and teacher. Reason step by step about chess positions, openings and defenses. Always structure your reasoning inside <reasoning>...</reasoning> tags before giving your final answer. Be thorough in your analysis."
        },
        {
            "role": "user",
            "content": "Explain the main ideas behind the Sicilian Defense (1.e4 c5). What are the key plans for both sides?"
        }
    ],
    temperature=0.3,
    max_tokens=5000,
    logprobs=True,
    top_logprobs=1
)

content = response.choices[0].message.content
print(f"Content est None : {content is None}")
print(f"Finish reason : {response.choices[0].finish_reason}")
print(f"Logprobs : {response.choices[0].logprobs is not None}")

if content:
    print(f"\nRéponse :\n{content[:1500]}")
else:
    print("\nPas de contenu dans la réponse. Vérifions la structure brute :")
    print(response.choices[0])

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid Authentication', 'type': 'authentication_error', 'param': None, 'code': None}}

### 3.2 Définition des instructions (échecs)

467 instructions couvrant : ouvertures, défenses, milieu de jeu, finales, stratégie, tactique, etc.

In [ ]:
chess_instructions = [
    # === THÉORIE D'OUVERTURE (30) ===
    # Ouvertures blancs
    "Explain the main ideas behind the Italian Game (1.e4 e5 2.Nf3 Nc6 3.Bc4). What are White's key plans?",
    "What is the Ruy Lopez (1.e4 e5 2.Nf3 Nc6 3.Bb5) and why has it been a top opening for centuries?",
    "Explain the Queen's Gambit (1.d4 d5 2.c4). What happens if Black accepts? What if Black declines?",
    "What are the main ideas of the London System (1.d4 2.Nf3 3.Bf4)? Why is it popular at club level?",
    "Explain the King's Gambit (1.e4 e5 2.f4). Is it sound at high level play? What are the risks and rewards?",
    "Describe the English Opening (1.c4). What type of positions does it lead to and what are White's plans?",
    "Explain the Scotch Game (1.e4 e5 2.Nf3 Nc6 3.d4). How does it differ strategically from the Italian Game?",
    "What is the Vienna Game (1.e4 e5 2.Nc3)? What are White's typical plans and piece placement?",
    "Explain the Catalan Opening (1.d4 Nf6 2.c4 e6 3.g3). Why is it a favorite of world champions?",
    "What is the Réti Opening (1.Nf3)? Explain its hypermodern philosophy.",
    "Explain the Four Knights Game (1.e4 e5 2.Nf3 Nc6 3.Nc3 Nf6). Is it drawish or can White fight for an advantage?",
    "What is the Bird's Opening (1.f4)? What are its strengths and weaknesses?",
    "Explain the Giuoco Piano vs the Evans Gambit. How does 4.b4 change the nature of the Italian Game?",
    "What is the Trompowsky Attack (1.d4 Nf6 2.Bg5)? When is it a good surprise weapon?",
    "Explain the Ponziani Opening (1.e4 e5 2.Nf3 Nc6 3.c3). What is White's idea with this modest move?",

    # Défenses noirs
    "Explain the Sicilian Defense (1.e4 c5). Why is it Black's most popular response to 1.e4?",
    "What is the French Defense (1.e4 e6)? Analyze its pawn structure, strengths, and weaknesses.",
    "Explain the Caro-Kann Defense (1.e4 c6). Why is it considered one of the most solid defenses?",
    "What is the Pirc Defense (1.e4 d6 2.d4 Nf6 3.Nc3 g6)? What type of player should choose it?",
    "Explain the Scandinavian Defense (1.e4 d5). What are the pros and cons of early queen development?",
    "What is the King's Indian Defense (1.d4 Nf6 2.c4 g6 3.Nc3 Bg7)? Explain the typical kingside attack.",
    "Explain the Nimzo-Indian Defense (1.d4 Nf6 2.c4 e6 3.Nc3 Bb4). Why do top GMs love this defense?",
    "What is the Slav Defense (1.d4 d5 2.c4 c6)? How does it compare to the Queen's Gambit Declined?",
    "Explain the Dutch Defense (1.d4 f5). What are its aggressive ideas and structural weaknesses?",
    "What is the Grünfeld Defense (1.d4 Nf6 2.c4 g6 3.Nc3 d5)? Explain how Black fights for the center.",
    "Explain the Alekhine Defense (1.e4 Nf6). What is the provocative idea behind this opening?",
    "What is the Benoni Defense (1.d4 Nf6 2.c4 c5)? Explain the pawn structure and typical plans.",
    "Explain the Petroff Defense (1.e4 e5 2.Nf3 Nf6). Why is it known as the most solid reply to 1.e4 e5?",
    "What is the Philidor Defense (1.e4 e5 2.Nf3 d6)? Is it passive or does Black have active plans?",
    "Explain the Owen Defense (1.e4 b6). What are the hypermodern ideas behind it?",

    # === ANALYSE DE POSITIONS (30) ===
    "After 1.e4 c5 2.Nf3 d6 3.d4 cxd4 4.Nxd4 Nf6 5.Nc3 a6 (Sicilian Najdorf), what are White's main options and plans?",
    "After 1.e4 e5 2.Nf3 Nc6 3.Bb5 a6 4.Ba4 Nf6 5.O-O Be7 6.Re1 b5 7.Bb3 O-O (Closed Ruy Lopez), what should White play and why?",
    "In the position after 1.d4 d5 2.c4 e6 3.Nc3 Nf6 4.Bg5 Be7 5.e3 O-O 6.Nf3 Nbd7, explain White's plan with the minority attack.",
    "After 1.e4 c5 2.Nf3 d6 3.d4 cxd4 4.Nxd4 Nf6 5.Nc3 g6 (Sicilian Dragon), why does White play 6.Be3 followed by f3, Qd2, and O-O-O?",
    "Analyze the position after 1.e4 e5 2.Nf3 Nc6 3.Bc4 Nf6 4.Ng5. Why is this attack on f7 dangerous and how should Black defend?",
    "After 1.d4 Nf6 2.c4 g6 3.Nc3 Bg7 4.e4 d6 5.Nf3 O-O 6.Be2 e5 7.O-O Nc6, explain the typical plans for both sides in the King's Indian.",
    "In the Sicilian Scheveningen (after 1.e4 c5 2.Nf3 d6 3.d4 cxd4 4.Nxd4 Nf6 5.Nc3 e6), explain the Keres Attack with 6.g4.",
    "After 1.e4 e6 2.d4 d5 3.Nc3 Bb4 (French Winawer), analyze 4.e5. What are the consequences for the pawn structure?",
    "Analyze the Exchange Variation of the Ruy Lopez: 1.e4 e5 2.Nf3 Nc6 3.Bb5 a6 4.Bxc6 dxc6. Why does White exchange and what are the resulting plans?",
    "After 1.d4 d5 2.c4 dxc4 (Queen's Gambit Accepted), explain why Black cannot hold the extra pawn and what White's plans are.",
    "In the Sveshnikov Sicilian after 1.e4 c5 2.Nf3 Nc6 3.d4 cxd4 4.Nxd4 Nf6 5.Nc3 e5 6.Ndb5 d6, explain why d5 is weak but Black is still OK.",
    "Analyze the position after 1.e4 e5 2.Nf3 Nc6 3.d4 exd4 4.Nxd4 Bc5 (Scotch Game). What are the critical continuations?",
    "After 1.d4 Nf6 2.c4 e6 3.Nc3 Bb4 4.Qc2 (Nimzo-Indian Classical), why does White play Qc2 and what does Black do?",
    "Explain the typical middlegame after 1.e4 c6 2.d4 d5 3.Nc3 dxe4 4.Nxe4 Bf5 (Caro-Kann Classical). What are the plans for both sides?",
    "After 1.e4 e5 2.Nf3 Nf6 3.Nxe5 d6 4.Nf3 Nxe4 (Petroff), analyze the position. Why is it considered equal?",
    "In the Benko Gambit (1.d4 Nf6 2.c4 c5 3.d5 b5), explain why Black sacrifices a pawn and what long-term compensation Black gets.",
    "Analyze the Fried Liver Attack: 1.e4 e5 2.Nf3 Nc6 3.Bc4 Nf6 4.Ng5 d5 5.exd5 Nxd5 6.Nxf7. Is it sound?",
    "After 1.e4 c5 2.c3 (Alapin Sicilian), explain White's plan. How should Black respond?",
    "In the English Attack against the Najdorf (6.Be3 followed by f3, Qd2, g4), explain the strategic battle between the two sides.",
    "Analyze the Marshall Attack: 1.e4 e5 2.Nf3 Nc6 3.Bb5 a6 4.Ba4 Nf6 5.O-O Be7 6.Re1 b5 7.Bb3 O-O 8.c3 d5. Why does Black sacrifice a pawn?",
    "After 1.d4 d5 2.c4 c6 3.Nf3 Nf6 4.Nc3 e6 5.e3 Nbd7 6.Bd3 dxc4 7.Bxc4 b5 (Semi-Slav Meran), explain the sharp play that follows.",
    "In the Advance French (1.e4 e6 2.d4 d5 3.e5), explain Black's typical plan with ...c5 and piece pressure on d4.",
    "Analyze the Smith-Morra Gambit (1.e4 c5 2.d4 cxd4 3.c3). Is the pawn sacrifice justified? What compensation does White get?",
    "After 1.d4 Nf6 2.c4 g6 3.Nc3 d5 4.cxd5 Nxd5 5.e4 Nxc3 6.bxc3 Bg7 (Grünfeld Exchange), explain why White's big center might be a target.",
    "In the Tarrasch Defense (1.d4 d5 2.c4 e6 3.Nc3 c5), explain the isolated queen pawn and whether it is a strength or weakness.",
    "Analyze the Grand Prix Attack against the Sicilian (1.e4 c5 2.Nc3 Nc6 3.f4). What are White's attacking ideas?",
    "After 1.e4 d5 2.exd5 Qxd5 3.Nc3 Qa5 (Scandinavian), explain Black's plan to develop while keeping the queen active.",
    "In the Closed Sicilian (1.e4 c5 2.Nc3 Nc6 3.g3), explain how this differs from the Open Sicilian and what type of game results.",
    "Analyze the position after 1.e4 e5 2.f4 exf4 3.Nf3 g5 (King's Gambit Accepted). What are White's attacking chances?",
    "After 1.d4 Nf6 2.c4 e6 3.g3 d5 4.Bg2 Be7 5.Nf3 O-O 6.O-O dxc4 (Open Catalan), explain how White recovers the pawn and maintains pressure.",

    # === COMPARAISONS ET STRATÉGIE (20) ===
    "Compare the Sicilian Najdorf and the Sicilian Dragon. Which is more aggressive and which is more solid?",
    "Compare the French Defense and the Caro-Kann. Both start with ...e6 or ...c6 against 1.e4 — what are the key differences?",
    "What are the differences between the Queen's Gambit Declined and the Slav Defense? Which gives Black more active play?",
    "Compare the King's Indian Defense and the Grünfeld Defense. Both involve ...g6 and ...Bg7 — how do the strategies differ?",
    "Italian Game vs Ruy Lopez: both develop the bishop early. What are the strategic differences between Bc4 and Bb5?",
    "Compare open games (1.e4 e5) vs semi-open games (1.e4 c5/e6/c6). What types of skills does each require?",
    "Compare 1.e4 and 1.d4 as first moves. What styles of play do they lead to?",
    "What is the difference between classical and hypermodern opening philosophy? Give examples of each.",
    "Compare the Nimzo-Indian and the Queen's Indian Defense. When should Black choose one over the other?",
    "Explain the concept of a gambit in chess openings. Compare the Queen's Gambit, King's Gambit, and Benko Gambit.",
    "What is the difference between playing for an attack and playing for positional advantage in the opening?",
    "Compare Anti-Sicilian systems (Alapin, Smith-Morra, Grand Prix) to the Open Sicilian. When should White avoid 2.Nf3 3.d4?",
    "Explain the concept of the pawn center in openings. Compare a broad center (e4+d4) vs a flexible center (c4+Nf3).",
    "Compare the Exchange variations in different openings (Exchange French, Exchange Slav, Exchange Ruy Lopez). Are they drawish?",
    "What are the pros and cons of early castling vs delayed castling in various openings?",
    "Compare the fianchetto bishop (g2 or g7) vs the classical bishop development (c4/f4 or c5/f5). When is each better?",
    "Explain how pawn structure in the opening determines middlegame and endgame plans. Give 3 examples.",
    "What is the difference between a sound gambit and an unsound gambit? Give examples from common openings.",
    "Compare rapid development openings (Italian, Scotch) vs slow strategic openings (English, Réti). What determines the choice?",
    "Explain the concept of transpositions in chess openings. Give 3 examples where different move orders reach the same position.",

    # === PÉDAGOGIE ET RECOMMANDATIONS (20) ===
    "What are the best openings for a complete beginner (under 1000 ELO) to learn? Explain your reasoning.",
    "What opening repertoire would you recommend for an intermediate player (1200-1500 ELO) playing White?",
    "What opening repertoire would you recommend for an intermediate player (1200-1500 ELO) playing Black against 1.e4?",
    "What opening repertoire would you recommend for an intermediate player (1200-1500 ELO) playing Black against 1.d4?",
    "Explain the concept of development and tempo in chess openings. Give examples of traps caused by poor development.",
    "What are the most common mistakes beginners make in the opening? List 5 with explanations.",
    "Explain the opening principles: control the center, develop pieces, castle early, connect rooks. Why does each matter?",
    "What is an opening trap? Describe 3 famous opening traps and how to avoid them.",
    "How should you study chess openings effectively? What is more important: memorizing moves or understanding ideas?",
    "Explain what happens when you ignore opening principles. Analyze the Scholar's Mate attempt (1.e4 e5 2.Qh5) and why it fails against correct play.",
    "What openings should you avoid as a beginner and why? Give 3 examples with explanations.",
    "How do you choose an opening that fits your playing style? Compare aggressive vs positional player preferences.",
    "Explain the importance of knowing the first 5-10 moves of your openings well. What happens when you leave theory?",
    "What is preparation in chess? How do strong players prepare their openings for specific opponents?",
    "Explain why the move order matters in openings. Give an example where playing moves in the wrong order loses.",
    "What are the most important endgame concepts that come directly from opening choices? Give 3 examples.",
    "How has computer analysis changed opening theory? Give examples of openings that were re-evaluated.",
    "Explain the concept of novelty in chess openings. Why do top players search for new moves?",
    "What is the best way to build an opening repertoire from scratch? Outline a step-by-step approach.",
    "Explain why understanding typical middlegame plans is more important than memorizing opening moves. Give examples.",
]

print(f"Nombre total de questions : {len(chess_instructions)}")

Nombre total de questions : 100


### 3.3 Fonction de génération avec logprobs

In [ ]:
import json
import time

SYSTEM_PROMPT = """You are a chess grandmaster and teacher. Reason step by step about chess positions, openings and defenses. Always structure your reasoning inside
<reasoning>...</reasoning> tags before giving your final answer. Be thorough in your analysis."""

def generate_teacher_response(client, instruction, temperature=0.3, max_retries=3):
  for attempt in range(max_retries):
      try:
          response = client.chat.completions.create(
              model=TEACHER_MODEL,
              messages=[
                  {"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": instruction}
              ],
              temperature=temperature,
              max_tokens=5000,
              logprobs=True,
              top_logprobs=1
          )

          content = response.choices[0].message.content
          logprobs = response.choices[0].logprobs

          if content and len(content) > 100:
              return {
                  "content": content,
                  "logprobs": logprobs,
                  "finish_reason": response.choices[0].finish_reason,
                  "tokens_used": response.usage.total_tokens
              }
          else:
              print(f"  Réponse trop courte, retry {attempt+1}/{max_retries}")

      except Exception as e:
          print(f"  Erreur: {e}, retry {attempt+1}/{max_retries}")
          time.sleep(5)

  return None

### 3.4 Génération Stage 1 — Basse température (τ = 0.3)

In [ ]:
stage1_data = []

print("=== STAGE 1 : Génération basse température (τ=0.3) ===\n")
for i, instruction in enumerate(chess_instructions):
    print(f"[{i+1}/{len(chess_instructions)}] {instruction[:70]}...")
    result = generate_teacher_response(client, instruction, temperature=0.3)

    if result:
        stage1_data.append({
            "instruction": instruction,
            "output": result["content"],
            "logprobs_data": [
                {"token": t.token, "logprob": t.logprob}
                for t in result["logprobs"].content
            ] if result["logprobs"] and result["logprobs"].content else [],
            "temperature": 0.3,
            "stage": "stage1"
        })
        print(f"  OK ({result['tokens_used']} tokens, {result['finish_reason']})")
    else:
        print(f"  ÉCHEC")

    time.sleep(2)

print(f"\n=== Stage 1 terminé : {len(stage1_data)}/{len(chess_instructions)} réponses ===")


=== STAGE 1 : Génération basse température (τ=0.3) ===

[1/100] Explain the main ideas behind the Italian Game (1.e4 e5 2.Nf3 Nc6 3.Bc...
  Erreur: Error code: 401 - {'error': {'message': 'Invalid Authentication', 'type': 'authentication_error', 'param': None, 'code': None}}, retry 1/3
  Erreur: Error code: 401 - {'error': {'message': 'Invalid Authentication', 'type': 'authentication_error', 'param': None, 'code': None}}, retry 2/3


KeyboardInterrupt: 

### 3.5 Génération Stage 2 — Haute température (τ = 0.9)

In [ ]:
stage2_data = []

print("=== STAGE 2 : Génération haute température (τ=0.9) ===\n")
for i, instruction in enumerate(chess_instructions):
    print(f"[{i+1}/{len(chess_instructions)}] {instruction[:70]}...")
    result = generate_teacher_response(client, instruction, temperature=0.9)

    if result:
        stage2_data.append({
            "instruction": instruction,
            "output": result["content"],
            "logprobs_data": [
                {"token": t.token, "logprob": t.logprob}
                for t in result["logprobs"].content
            ] if result["logprobs"] and result["logprobs"].content else [],
            "temperature": 0.9,
            "stage": "stage2"
        })
        print(f"  OK ({result['tokens_used']} tokens, {result['finish_reason']})")
    else:
        print(f"  ÉCHEC")

    time.sleep(2)

print(f"\n=== Stage 2 terminé : {len(stage2_data)}/{len(chess_instructions)} réponses ===")

### 3.6 Sauvegarde des données brutes

In [ ]:
with open("stage1_raw.json", "w") as f:
    json.dump(stage1_data, f, ensure_ascii=False, indent=2)

with open("stage2_raw.json", "w") as f:
    json.dump(stage2_data, f, ensure_ascii=False, indent=2)

print(f"Stage 1 : {len(stage1_data)} exemples sauvegardés dans stage1_raw.json")
print(f"Stage 2 : {len(stage2_data)} exemples sauvegardés dans stage2_raw.json")

---
## Phase 4 : Divergence-Aware Sampling (DAS)

### Principe

Le DAS analyse la **divergence phrase par phrase** entre le Teacher et le Student :
- **Teacher Sentence** : le Teacher est confiant, le Student échoue → **valeur pédagogique forte**
- **Shared Sentence** : les deux sont d'accord → neutre
- **Student Sentence** : le Student est trop confiant → bruit nuisible

### Adaptation pour notre domaine (échecs)

Le modèle étudiant (Qwen3-4B) ne connaît quasiment rien aux échecs. Par conséquent, **toutes les phrases** ont une divergence Teacher >> Student, rendant le filtre DAS standard inefficace (il garderait tout).

**Notre solution** : filtrer sur la **confiance du Teacher seul** (moyenne géométrique de ses probabilités par phrase). Si le Teacher est incertain sur sa propre réponse, la donnée est probablement de mauvaise qualité et on la rejette.

### 4.1 Chargement des données et du modèle étudiant

In [ ]:
# avec les 467 données

!pip install -U bitsandbytes>=0.46.1 accelerate transformers nltk

In [ ]:
import json
import torch
import numpy as np
import nltk
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files

nltk.download('punkt_tab', quiet=True)

uploaded = files.upload()  # stage1_raw.json et stage2_raw.json

stage1_data = json.load(open("stage1_raw.json", encoding="utf-8"))
stage2_data = json.load(open("stage2_raw.json", encoding="utf-8"))
print(f"Stage 1 : {len(stage1_data)} exemples")
print(f"Stage 2 : {len(stage2_data)} exemples")

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"

print(f"Chargement de {MODEL_ID} en float16...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Modèle chargé !")

### 4.2 Calcul des scores DAS (Teacher vs Student)

In [ ]:
def compute_das_scores(example, tokenizer, model):
    instruction = example["instruction"]
    teacher_text = example["output"]
    teacher_logprobs = example["logprobs_data"]

    messages = [{"role": "user", "content": instruction}]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    full_input_str = prompt_str + teacher_text

    inputs = tokenizer(full_input_str, return_tensors="pt", truncation=True,max_length=2048).to(model.device)
    prompt_tokens_len = len(tokenizer(prompt_str, add_special_tokens=False)["input_ids"])

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    shift_logits = logits[0, :-1, :]
    shift_labels = inputs["input_ids"][0, 1:]

    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    token_losses = loss_fct(shift_logits, shift_labels)
    token_logprobs_student = -token_losses.cpu().numpy()

    del outputs, logits, shift_logits, token_losses, inputs
    torch.cuda.empty_cache()

    response_logprobs_student = token_logprobs_student[prompt_tokens_len - 1:]
    response_token_ids = shift_labels[prompt_tokens_len - 1:].cpu().numpy()
    del shift_labels

    sentences = nltk.tokenize.sent_tokenize(teacher_text)
    results = []
    openai_cursor = 0
    qwen_cursor = 0

    for sent in sentences:
        current_accum = ""
        sent_teacher_logprobs = []
        while openai_cursor < len(teacher_logprobs):
            t_data = teacher_logprobs[openai_cursor]
            sent_teacher_logprobs.append(t_data["logprob"])
            current_accum += t_data["token"]
            openai_cursor += 1
            if len(current_accum) >= len(sent):
                break
        p_teacher = np.exp(np.mean(sent_teacher_logprobs)) if sent_teacher_logprobs else 0

        current_accum_qwen = ""
        sent_student_logprobs = []
        while qwen_cursor < len(response_token_ids):
            tid = response_token_ids[qwen_cursor]
            token_str = tokenizer.decode([tid])
            sent_student_logprobs.append(response_logprobs_student[qwen_cursor])
            current_accum_qwen += token_str
            qwen_cursor += 1
            if len(current_accum_qwen) >= len(sent):
                break
        p_student = np.exp(np.mean(sent_student_logprobs)) if sent_student_logprobs else 0

        results.append({
            "sentence": sent,
            "p_teacher": float(p_teacher),
            "p_student": float(p_student),
            "divergence": float(p_teacher - p_student)
        })

    return results

### 4.3 Filtrage adapté : confiance du Teacher

Comme le DAS standard garde 100% des exemples (divergence toujours positive), on filtre à la place sur `avg_p_teacher` ≥ seuil.

In [ ]:
# === Seuil DAS adapté ===
P_TEACHER_MIN = 0.3  # confiance min du teacher

def filter_with_das(data, tokenizer, model, stage_name):
    """Filtrage adapté : on garde les exemples où le Teacher est confiant."""
    print(f"\n=== DAS Filtering : {stage_name} ({len(data)} exemples) ===")
    print(f"    Seuil : p_teacher_min > {P_TEACHER_MIN}\n")

    kept = []
    rejected = []

    for i, example in enumerate(data):
        try:
            scores = compute_das_scores(example, tokenizer, model)
            avg_p_teacher = np.mean([s["p_teacher"] for s in scores]) if scores else 0

            example["das_scores"] = scores
            example["avg_p_teacher"] = float(avg_p_teacher)

            if avg_p_teacher >= P_TEACHER_MIN:
                kept.append(example)
                print(f"  [{i+1}/{len(data)}] p_teacher={avg_p_teacher:.4f} → KEEP")
            else:
                rejected.append(example)
                print(f"  [{i+1}/{len(data)}] p_teacher={avg_p_teacher:.4f} → SKIP (teacher incertain)")

        except Exception as e:
            print(f"  [{i+1}/{len(data)}] Erreur: {e}")

    print(f"\n{stage_name} : {len(kept)}/{len(data)} gardés")
    print(f"  Rejetés (teacher incertain) : {len(rejected)}")

    return kept

stage1_filtered = filter_with_das(stage1_data, tokenizer, model, "Stage 1")
stage2_filtered = filter_with_das(stage2_data, tokenizer, model, "Stage 2")

### 4.4 Visualisation et sauvegarde des résultats

In [ ]:
import matplotlib.pyplot as plt

# Sauvegarder les données filtrées
with open("stage1_filtered.json", "w", encoding="utf-8") as f:
    json.dump(stage1_filtered, f, ensure_ascii=False, indent=2)
with open("stage2_filtered.json", "w", encoding="utf-8") as f:
    json.dump(stage2_filtered, f, ensure_ascii=False, indent=2)

print(f"Stage 1 : {len(stage1_filtered)}/{len(stage1_data)} gardés")
print(f"Stage 2 : {len(stage2_filtered)}/{len(stage2_data)} gardés")

# Histogramme de la confiance Teacher
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, name in [(axes[0], stage1_data, "Stage 1"), (axes[1], stage2_data, "Stage 2")]:
    pts = [ex["avg_p_teacher"] for ex in data if "avg_p_teacher" in ex]
    ax.hist(pts, bins=20, edgecolor='black')
    ax.axvline(x=P_TEACHER_MIN, color='red', linestyle='--', label=f'Seuil ({P_TEACHER_MIN})')
    ax.set_title(f"{name} - Confiance Teacher")
    ax.set_xlabel("avg p_teacher")
    ax.set_ylabel("Nombre d'exemples")
    ax.legend()
plt.tight_layout()
plt.savefig("das_distribution.png")
plt.show()
print("Histogramme sauvegardé dans das_distribution.png")

In [ ]:
from google.colab import files
files.download("stage1_filtered.json")
files.download("stage2_filtered.json")
files.download("das_distribution.png")

---
## Phase 5 : Configuration et entraînement

### Architecture d'entraînement (Temperature-Scheduled Learning)

| Stage | Données | Température | Objectif |
|-------|---------|-------------|----------|
| **Stage 1** | `chess_stage1.json` | τ = 0.3 (basse) | Apprendre les fondamentaux avec des réponses stables |
| **Stage 2** | `chess_stage2.json` | τ = 0.9 (haute) | Diversifier le raisonnement avec des réponses variées |

Le Stage 2 **charge l'adapter LoRA du Stage 1** pour continuer l'apprentissage.

### 5.1 Libération mémoire et préparation des données

In [ ]:
# Upload des données brutes depuis votre machine
from google.colab import files

print("Uploadez stage1_raw.json et stage2_raw.json")
uploaded = files.upload()

In [ ]:
# Libérer le modèle DAS de la mémoire GPU (si chargé)
import torch
import gc
try:
    del model, tokenizer
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Mémoire GPU libérée")

In [ ]:
# Conversion des données raw au format ShareGPT pour Llama-Factory
import json

SYSTEM_PROMPT = (
    "You are a chess grandmaster and teacher. Reason step by step about chess "
    "positions, openings and defenses. Always structure your reasoning inside "
    "<reasoning>...</reasoning> tags before giving your final answer. "
    "Be thorough in your analysis."
)

def convert_to_sharegpt(examples):
    """Convertit au format ShareGPT attendu par Llama-Factory."""
    converted = []
    for ex in examples:
        converted.append({
            "conversations": [
                {"from": "system", "value": SYSTEM_PROMPT},
                {"from": "human", "value": ex["instruction"]},
                {"from": "gpt", "value": ex["output"]}
            ]
        })
    return converted

# Charger les données raw (le filtre DAS gardait tout, cf. Phase 4)
stage1_data = json.load(open("stage1_raw.json", encoding="utf-8"))
stage2_data = json.load(open("stage2_raw.json", encoding="utf-8"))

# Convertir
chess_stage1 = convert_to_sharegpt(stage1_data)
chess_stage2 = convert_to_sharegpt(stage2_data)

# Sauvegarder
with open("chess_stage1.json", "w", encoding="utf-8") as f:
    json.dump(chess_stage1, f, ensure_ascii=False, indent=2)
with open("chess_stage2.json", "w", encoding="utf-8") as f:
    json.dump(chess_stage2, f, ensure_ascii=False, indent=2)

print(f"chess_stage1.json : {len(chess_stage1)} exemples (format ShareGPT)")
print(f"chess_stage2.json : {len(chess_stage2)} exemples (format ShareGPT)")
print()
print("Exemple de format :")
print(json.dumps(chess_stage1[0]["conversations"][:2], indent=2, ensure_ascii=False))

### 5.2 Configuration de Llama-Factory

In [ ]:
import json
import os
import subprocess
import shutil

# --- Installer Llama-Factory si pas déjà fait ---
LLAMA_FACTORY_DIR = "LLaMA-Factory"

if not os.path.exists(LLAMA_FACTORY_DIR):
    print("Installation de Llama-Factory...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/hiyouga/LLaMA-Factory.git"], check=True)
    subprocess.run(["pip", "install", "-e", ".[torch,bitsandbytes]"], cwd=LLAMA_FACTORY_DIR, check=True)
    print("Llama-Factory installé !")
else:
    print(f"Llama-Factory déjà présent dans {LLAMA_FACTORY_DIR}/")

DATA_DIR = os.path.join(LLAMA_FACTORY_DIR, "data")

# --- dataset_info.json : enregistrement des datasets ---
dataset_info = {
    "chess_stage1": {
        "file_name": "chess_stage1.json",
        "formatting": "sharegpt",
        "columns": {
            "messages": "conversations"
        }
    },
    "chess_stage2": {
        "file_name": "chess_stage2.json",
        "formatting": "sharegpt",
        "columns": {
            "messages": "conversations"
        }
    }
}

# Copier les datasets
for fname in ["chess_stage1.json", "chess_stage2.json"]:
    shutil.copy(fname, os.path.join(DATA_DIR, fname))
    print(f"Copié : {fname} → {DATA_DIR}/")

# Fusionner avec le dataset_info.json existant de Llama-Factory
existing_info_path = os.path.join(DATA_DIR, "dataset_info.json")
if os.path.exists(existing_info_path):
    with open(existing_info_path, encoding="utf-8") as f:
        existing_info = json.load(f)
    existing_info.update(dataset_info)
else:
    existing_info = dataset_info

with open(existing_info_path, "w", encoding="utf-8") as f:
    json.dump(existing_info, f, indent=2, ensure_ascii=False)
print(f"dataset_info.json mis à jour dans {DATA_DIR}/")

In [ ]:
# --- Configuration YAML Stage 1 ---
stage1_config = """### Stage 1 : Fine-tuning LoRA — données basse température (échecs)

### Modèle
model_name_or_path: unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit
trust_remote_code: true

### Méthode
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05

### Dataset
dataset: chess_stage1
template: qwen3
cutoff_len: 2048

### Entraînement
per_device_train_batch_size: 2
gradient_accumulation_steps: 4
learning_rate: 2.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
gradient_checkpointing: true

### Précision (T4 = fp16)
fp16: true

### Logging & sauvegarde
logging_steps: 10
save_steps: 100
save_total_limit: 2
report_to: none

### Sortie
output_dir: saves/qwen3-4b-chess/lora/stage1
overwrite_output_dir: true
"""

with open("stage1_config.yaml", "w", encoding="utf-8") as f:
    f.write(stage1_config)
print("stage1_config.yaml créé")
print()
print(stage1_config)

In [ ]:
# --- Configuration YAML Stage 2 ---
# Charge l'adapter LoRA du Stage 1
stage2_config = """### Stage 2 : Fine-tuning LoRA — données haute température (échecs)
### Charge l'adapter du Stage 1

### Modèle
model_name_or_path: unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit
adapter_name_or_path: saves/qwen3-4b-chess/lora/stage1
trust_remote_code: true

### Méthode
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05

### Dataset
dataset: chess_stage2
template: qwen3
cutoff_len: 2048

### Entraînement (lr plus bas pour stage 2)
per_device_train_batch_size: 2
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
gradient_checkpointing: true

### Précision (T4 = fp16)
fp16: true

### Logging & sauvegarde
logging_steps: 10
save_steps: 100
save_total_limit: 2
report_to: none

### Sortie
output_dir: saves/qwen3-4b-chess/lora/stage2
overwrite_output_dir: true
"""

with open("stage2_config.yaml", "w", encoding="utf-8") as f:
    f.write(stage2_config)
print("stage2_config.yaml créé")
print()
print(stage2_config)

### 5.3 Entraînement Stage 1 (basse température)

> **Durée estimée** : ~30-45 min sur T4

In [ ]:
!cd LLaMA-Factory && llamafactory-cli train ../stage1_config.yaml

### 5.4 Entraînement Stage 2 (haute température)

Le Stage 2 charge automatiquement l'adapter LoRA entraîné au Stage 1 via `adapter_name_or_path`.

> **Durée estimée** : ~20-30 min sur T4

In [ ]:
!cd LLaMA-Factory && llamafactory-cli train ../stage2_config.yaml

---
## Phase 7 : Vérification des résultats

In [ ]:
import os
import json

# Vérifier les checkpoints
for stage in ["stage1", "stage2"]:
    checkpoint_dir = f"saves/qwen3-4b-chess/lora/{stage}"
    if os.path.exists(checkpoint_dir):
        files = os.listdir(checkpoint_dir)
        print(f"\n{stage} checkpoints ({checkpoint_dir}):")
        for f in sorted(files):
            path = os.path.join(checkpoint_dir, f)
            size = os.path.getsize(path) if os.path.isfile(path) else "dir"
            print(f"  {f} ({size})")
    else:
        print(f"\n{stage}: pas de checkpoint trouvé à {checkpoint_dir}")

In [ ]:
# Tracer les courbes de loss
import json
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, stage in zip(axes, ["stage1", "stage2"]):
    log_path = f"saves/qwen3-4b-chess/lora/{stage}/trainer_log.jsonl"
    if os.path.exists(log_path):
        steps, losses = [], []
        with open(log_path) as f:
            for line in f:
                entry = json.loads(line)
                if "loss" in entry:
                    steps.append(entry.get("current_steps", len(steps)))
                    losses.append(entry["loss"])
        ax.plot(steps, losses)
        ax.set_title(f"{stage.upper()} - Courbe de Loss")
        ax.set_xlabel("Steps")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f"{stage.upper()} - Pas de log trouvé")

plt.tight_layout()
plt.savefig("training_loss_curves.png", dpi=150)
plt.show()
print("Courbes sauvegardées dans training_loss_curves.png")

---
## Phase 8 : Test du modèle distillé

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Charger le modèle de base
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
ADAPTER_PATH = "saves/qwen3-4b-chess/lora/stage2"  # adapter final (après les 2 stages)

print("Chargement du modèle de base...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    trust_remote_code=True
)

print("Chargement de l'adapter LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Modèle distillé chargé !")

In [ ]:
# Test sur des prompts d'échecs
test_prompts = [
    "Explain the main ideas behind the Sicilian Defense (1.e4 c5). What are the key plans for both sides?",
    "What is the best strategy in a King and Pawn endgame when you have an extra pawn?",
    "Explain the concept of pawn structure and why it matters in chess strategy.",
    "What are the key principles of piece development in the opening?",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a chess grandmaster and teacher. Reason step by step about chess positions, openings and defenses. Always structure your reasoning inside <reasoning>...</reasoning> tags before giving your final answer."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    print(f"\n{'='*80}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*80}")
    print(response[:500])
    if len(response) > 500:
        print(f"... ({len(response)} caractères au total)")
    print()

---
## Phase 9 : Évaluation quantitative

Comparaison du modèle **avant** (base) et **après** distillation sur un ensemble de test.

In [ ]:
# Évaluation : comparer base vs distillé
# On teste sur des questions d'échecs non vues pendant l'entraînement

eval_questions = [
    "What is a fianchetto and when should you use it?",
    "Explain the difference between a tactical and a positional player.",
    "What are the main ideas behind the Caro-Kann Defense?",
    "How do you evaluate whether to trade pieces or keep them on the board?",
    "Explain the concept of prophylaxis in chess.",
]

# Fonction d'évaluation simple : longueur de raisonnement + présence de <reasoning>
def evaluate_response(model, tokenizer, prompt):
    messages = [
        {"role": "system", "content": "You are a chess grandmaster and teacher. Reason step by step. Use <reasoning>...</reasoning> tags."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.3, do_sample=True)
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    has_reasoning = "<reasoning>" in response and "</reasoning>" in response
    reasoning_len = 0
    if has_reasoning:
        import re
        match = re.search(r"<reasoning>(.*?)</reasoning>", response, re.DOTALL)
        if match:
            reasoning_len = len(match.group(1).split())
    
    return {
        "response": response,
        "has_reasoning": has_reasoning,
        "reasoning_words": reasoning_len,
        "total_words": len(response.split())
    }

print("=== Évaluation du modèle distillé ===\n")
results = []
for q in eval_questions:
    r = evaluate_response(model, tokenizer, q)
    results.append(r)
    print(f"Q: {q[:60]}...")
    print(f"  Reasoning: {'Oui' if r['has_reasoning'] else 'Non'} | "
          f"Mots reasoning: {r['reasoning_words']} | Total: {r['total_words']}")
    print()

# Résumé
reasoning_rate = sum(1 for r in results if r["has_reasoning"]) / len(results)
avg_reasoning = sum(r["reasoning_words"] for r in results) / len(results)
print(f"\n=== Résumé ===")
print(f"Taux de réponses avec <reasoning>: {reasoning_rate:.0%}")
print(f"Longueur moyenne du raisonnement: {avg_reasoning:.0f} mots")